# Optional real-time CNN control / YOLOX-S

Anchor-free one-stage control. The notebook intentionally requires the official YOLOX repository; it never substitutes a differently licensed package.

License and exact weight provenance are recorded in `LICENSES.md` and each run manifest. Approximate GPU requirements depend strongly on resolution, batch size, AMP, and the active Colab GPU.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

## Editable experiment configuration

In [ ]:
MODEL_ID = "yolox_s"
DATASET_TRACK = "2class"
IMAGE_SIZE = 1024
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
NUM_EPOCHS = 100
SEED = 42
USE_AMP = True
RESUME_RUN_ID = None
RUN_HYPERPARAMETER_SEARCH = False
print(dict(MODEL_ID=MODEL_ID, DATASET_TRACK=DATASET_TRACK, IMAGE_SIZE=IMAGE_SIZE, EFFECTIVE_BATCH_SIZE=EFFECTIVE_BATCH_SIZE))

## Dataset validation

Validation checks image existence, dimensions, category IDs, bbox coordinates, zero-area boxes, and class coverage. Statistics expose class counts, size distributions, and objects per image.

In [ ]:
from src.notebook_utils import preflight_dataset
report = preflight_dataset(paths, DATASET_TRACK, minimum_free_gb=0 if SMOKE_TEST else 5)
print(report)
report.raise_for_errors()


## Model construction and introspection

The training command saves architecture, parameter totals, trainable/frozen totals, runtime config, and environment. After a first checkpoint, use notebook 08 for feature shapes, stage strides, FLOPs/MACs, and actual module names.

In [ ]:
if SMOKE_TEST:
    print("SMOKE_TEST: optional YOLOX clone and integration are not executed.")
else:
    raise RuntimeError(
        "YOLOX-S is an optional control and its adapter is not implemented. "
        "Do not treat this notebook as a completed training path."
    )
